Этапы:

0. Установка библиотек
1. Распознавание PDF-файлов через PPStructureV3 (лучшие результаты)
2. Разделением документов на БЛОКИ, производится по объекту 'paragraph_title'. Собираем объекты 'formula', 'text', 'paragraph_title'.
3. Извлекаем ключевые слова из объектов 'text'
4. (ВРЕМЕННО УБРАНО) Создаём граф знаний RDFLIB по ключевым словам - блокам - текстам
5. (УБРАНО) Конвертируем формулы из LaTex в MathML (https://math.nist.gov/~BMiller/LaTeXML/)
6. (УБРАНО) Добавляем в индекс BehroozMansouri/TangentCFT. Создаём индекс формула - блок текста.
5. Производим поиск через MathBERT
6. Производим фильтрацию (по графу знаний планируется, сейчас просто сравнение ключевых слов)

### 0. Установка библиотек

In [ ]:
!pip3 install rdflib Mathics-omnibus[full] paddleocr[all] keybert
!pip install sentence-transformers
!pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install --upgrade jupyter ipywidgets

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/nvidia-cuda-nvrtc-cu12/nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl (23.7 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/nvidia-cuda-runtime-cu12/nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (897 kB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/nvidia-cuda-cupti-cu12/nvidia_cuda_cupti_cu12-12.6.80-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.9 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/nvidia-cudnn-cu12/nvidia_cudnn_cu12-9.5.1.17-py3-none-manylinux_2_28_x86_64.whl (571.0 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/nvidia-cublas-cu12/nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (393.1 MB)
  Using cached https://paddle-whl.bj.bcebos.com/stable/cu126/nvidia-cufft-c

### 1. Распознавание PDF-файлов через PPStructureV3 (лучшие результаты)

In [40]:
# https://arxiv.org/pdf/2507.05595
from paddleocr import PPStructureV3

pipeline = PPStructureV3(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_chart_recognition=False,
    use_seal_recognition=False,
    lang="ru"
)

/home/alexey/test/mipt_mag_diploma/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-DocBlockLayout', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-DocBlockLayout`.
Creating model: ('PP-DocLayout_plus-L', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-DocLayout_plus-L`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: 

In [1]:
from pathlib import Path
import os
import json

DATA_PATH="../../data/do_not_upload"
DATA_PATH_OCR=os.path.join(DATA_PATH, "ocr_data")
PDF_GLOB="*.pdf"
Path(DATA_PATH_OCR).mkdir(exist_ok=True)
documents = []

In [42]:
for pdf in list(Path(DATA_PATH).glob(PDF_GLOB)):
    name = str(pdf).split(os.path.sep)[-1]
    data_loc = os.path.join(DATA_PATH, name.split('.')[0])
    print(f"Processing {name}...")
    output = pipeline.predict(input=str(pdf))
    #res.print()
    print(f"Done. Saving to {data_loc}...")
    for res in output:
        res.save_to_json(save_path=data_loc)
        res.save_to_markdown(save_path=data_loc)
        res.save_to_img(save_path=data_loc)
    #documents.append(output)

Processing 03-12.pdf...


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('eslav_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/alexey/.paddlex/official_models/eslav_PP-OCRv5_mobile_rec`.


Done. Saving to ../../data/do_not_upload/03-12...
Processing 07-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/07-3-5-3...
Processing 06-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/06-3-5-3...
Processing 01-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/01-3-5-3...
Processing 10-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/10-3-5-3...
Processing 04-12.pdf...
Done. Saving to ../../data/do_not_upload/04-12...
Processing 03-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/03-3-5-3...
Processing 02-12.pdf...
Done. Saving to ../../data/do_not_upload/02-12...
Processing 11-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/11-3-5-3...
Processing 02-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/02-3-5-3...
Processing 05-12.pdf...
Done. Saving to ../../data/do_not_upload/05-12...
Processing 08-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/08-3-5-3...
Processing 09-3-5-3.pdf...
Done. Saving to ../../data/do_not_upload/09-3-5-3...
Processi

In [2]:
# Reread again
# I forgot to save it to variable...
documents = []
for pdf in list(Path(DATA_PATH).glob(PDF_GLOB)):
    name = str(pdf).split(os.path.sep)[-1]
    data_loc = os.path.join(DATA_PATH, name.split('.')[0])
    print(f"Processing {name}...")
    json_loc = list(Path(f"{data_loc}/").glob(f"{name.split('.')[0]}_*_res.json"))
    doc = list()
    for it in json_loc:
        print(it)
        with open(it, 'r') as f:
            doc.append(json.load(f))
            doc[-1]['source'] = str(it)
    documents.append(doc)

Processing 03-12.pdf...
../../data/do_not_upload/03-12/03-12_1_res.json
../../data/do_not_upload/03-12/03-12_5_res.json
../../data/do_not_upload/03-12/03-12_4_res.json
../../data/do_not_upload/03-12/03-12_6_res.json
../../data/do_not_upload/03-12/03-12_0_res.json
../../data/do_not_upload/03-12/03-12_3_res.json
../../data/do_not_upload/03-12/03-12_2_res.json
Processing 07-3-5-3.pdf...
../../data/do_not_upload/07-3-5-3/07-3-5-3_5_res.json
../../data/do_not_upload/07-3-5-3/07-3-5-3_6_res.json
../../data/do_not_upload/07-3-5-3/07-3-5-3_1_res.json
../../data/do_not_upload/07-3-5-3/07-3-5-3_2_res.json
../../data/do_not_upload/07-3-5-3/07-3-5-3_3_res.json
../../data/do_not_upload/07-3-5-3/07-3-5-3_4_res.json
../../data/do_not_upload/07-3-5-3/07-3-5-3_0_res.json
Processing 06-3-5-3.pdf...
../../data/do_not_upload/06-3-5-3/06-3-5-3_7_res.json
../../data/do_not_upload/06-3-5-3/06-3-5-3_9_res.json
../../data/do_not_upload/06-3-5-3/06-3-5-3_2_res.json
../../data/do_not_upload/06-3-5-3/06-3-5-3_6_r

### 2. Разделением документов на БЛОКИ, производится по объекту 'paragraph_title'. Собираем объекты 'formula', 'text', 'paragraph_title'.

In [14]:
class doc_data_blocks:
    pass
class data_block:
    def __init__(self, source_obj: doc_data_blocks):
        self.source_obj = source_obj
        self.text = ""
        self.formulas = list[str]()
        self.keywords = list[str]()
    def __str__(self):
        return f"Block of {self.source_obj.location[0]['input_path']}."
    def __repr__(self):
        return f"Block of {self.source_obj.location[0]['input_path']}."
class doc_data_blocks:
    def __init__(self, location):
        self.location = location
        self.blocks = list[data_block]()
    def __str__(self):
        return f"{self.location[0]['input_path']} with {len(self.blocks)} blocks."

In [16]:
docs_data = list[doc_data_blocks]()
for doc in documents:
    x = doc_data_blocks(doc)
    docs_data.append(x)
    block_str = data_block(x)
    paragraph_title_first_found = False
    previous_block = None
    for page in doc:
        for node in page["parsing_res_list"]:
            # If it paragraph_title, start new block
            if node["block_label"] == "formula":
                block_str.formulas.append(node["block_content"])
            if node["block_label"] == "paragraph_title":
                paragraph_title_first_found = True
                if previous_block == "paragraph_title":
                    block_str.text += f" {node['block_content']}"
                    pass
                else:
                    ## Previous text too small, add to it manually
                    #if len(x.text_blocks) >1 and len(x.text_blocks[-1]) < 100:
                    #    block_str += " "
                    #    block_str += node["block_content"]
                    #else:
                    if block_str.text != "":
                        x.blocks.append(block_str)
                        block_str = data_block(x)
                    block_str.text += node["block_content"]
            elif node["block_label"] == "text":
                # Skip until first paragraph_title met
                if paragraph_title_first_found:
                    block_str.text += f" {node['block_content']}"
            else:
                pass
            previous_block = node["block_content"]

In [5]:
for x in docs_data:
    for i in x.blocks:
        #print(i.text)
        #print(i.formulas)
        print("-----")

-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----
-----


### 3. Извлекаем ключевые слова из объектов 'text'

In [6]:
from keybert import KeyBERT
kw_model = KeyBERT()

In [18]:
for doc in docs_data:
    print(f"=={doc.location[0]["input_path"]}")
    for block in doc.blocks:
        # 2 words has more info
        keywords = kw_model.extract_keywords(block.text, keyphrase_ngram_range=(2, 2))
        #print(keywords)
        block.keywords=keywords

==../../data/do_not_upload/03-12.pdf
==../../data/do_not_upload/07-3-5-3.pdf
==../../data/do_not_upload/06-3-5-3.pdf
==../../data/do_not_upload/01-3-5-3.pdf
==../../data/do_not_upload/10-3-5-3.pdf
==../../data/do_not_upload/04-12.pdf
==../../data/do_not_upload/03-3-5-3.pdf
==../../data/do_not_upload/02-12.pdf
==../../data/do_not_upload/11-3-5-3.pdf
==../../data/do_not_upload/02-3-5-3.pdf
==../../data/do_not_upload/05-12.pdf
==../../data/do_not_upload/08-3-5-3.pdf
==../../data/do_not_upload/09-3-5-3.pdf
==../../data/do_not_upload/04-3-5-3.pdf
==../../data/do_not_upload/05-3-5-3.pdf


### (ВРЕМЕННО УБРАНО) 4. Создаём граф знаний RDFLIB по ключевым словам - блокам - текстам

In [ ]:
# https://rdflib.readthedocs.io/en/stable/gettingstarted/#a-more-extensive-example

from rdflib import Graph, Literal, RDF, URIRef
# rdflib knows about quite a few popular namespaces, like W3C ontologies, schema.org etc.
from rdflib.namespace import FOAF , XSD

# Create a Graph
g = Graph()

# Create an RDF URI node to use as the subject for multiple triples
donna = URIRef("http://example.org/donna")

# Add triples using store's add() method.
g.add((donna, RDF.type, FOAF.Person))
g.add((donna, FOAF.nick, Literal("donna", lang="en")))
g.add((donna, FOAF.name, Literal("Donna Fales")))
g.add((donna, FOAF.mbox, URIRef("mailto:donna@example.org")))

# Add another person
ed = URIRef("http://example.org/edward")

# Add triples using store's add() method.
g.add((ed, RDF.type, FOAF.Person))
g.add((ed, FOAF.nick, Literal("ed", datatype=XSD.string)))
g.add((ed, FOAF.name, Literal("Edward Scissorhands")))
g.add((ed, FOAF.mbox, Literal("e.scissorhands@example.org", datatype=XSD.anyURI)))

# Iterate over triples in store and print them out.
print("--- printing raw triples ---")
for s, p, o in g:
    print((s, p, o))

# For each foaf:Person in the store, print out their mbox property's value.
print("--- printing mboxes ---")
for person in g.subjects(RDF.type, FOAF.Person):
    for mbox in g.objects(person, FOAF.mbox):
        print(mbox)

# Bind the FOAF namespace to a prefix for more readable output
g.bind("foaf", FOAF)

# print all the data in the Notation3 format
print("--- printing mboxes ---")
print(g.serialize(format='n3'))

### 5. Производим поиск через MathBERT

In [8]:
from sentence_transformers import SentenceTransformer
# https://huggingface.co/tbs17/MathBERT
math_ber_model = SentenceTransformer("tbs17/MathBERT")

def math_similarity(a: list[str], b: list[str]):
    # https://www.sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html
    predictions = a
    references = b
    embeddings1 = math_ber_model.encode(predictions)
    embeddings2 = math_ber_model.encode(references)
    similarities = math_ber_model.similarity(embeddings1, embeddings2)
    # Output the pairs with their score
    #for idx_i, sentence1 in enumerate(predictions):
    #    print(sentence1)
    #    for idx_j, sentence2 in enumerate(references):
    #        print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")
    return similarities

No sentence-transformers model found with name tbs17/MathBERT. Creating a new one with mean pooling.


In [20]:
results = math_similarity(["a+b", "a*b+c*b"], ["b+a", "a*(b+c)", "k*(a+c)"])
results

tensor([[0.8977, 0.6999, 0.5350],
        [0.6765, 0.8351, 0.6820]])

In [19]:
test_doc_data = docs_data[0]
test_doc_data.blocks[0]

Block of ../../data/do_not_upload/03-12.pdf.

In [20]:
import difflib
# Ineffective, just try
def similar_docs(doc_data_in: doc_data_blocks, doc_data_blocks: list[doc_data_blocks]):
    sim_doc_data_blocks = []
    sim_doc_data_blocks_prob = []
    sim_doc_data_blocks_test = []
    for i1 in range(len(doc_data_in.blocks)):
        print(f"Processing block {i1+1} of {len(doc_data_in.blocks)}")
        block = doc_data_in.blocks[i1]
        for formula in block.formulas:
            # Too many time, lets just leave first found
            found = False
            for other_block in doc_data_blocks:
                if other_block == doc_data_in:
                    continue
                if other_block in sim_doc_data_blocks:
                    continue
                for other_block in other_block.blocks:
                    if not other_block.formulas:
                        continue
                    sim = math_similarity([formula], other_block.formulas)
                    for i in range(len(sim[0])):
                        x = sim[0][i]
                        if x > 0.65:
                            #print(f"Found similar formula in doc {other_block.source_obj.location[0]['input_path']}")
                            sim_doc_data_blocks.append(other_block)
                            sim_doc_data_blocks_prob.append(x)
                            sim_doc_data_blocks_test.append(block)
                            similar_words = []
                            other_keywords = [word[0] for word in other_block.keywords]
                            for word1 in block.keywords:
                                similar_words += difflib.get_close_matches(word1[0], other_keywords, cutoff=0.45)
                            if len(similar_words)>0:
                                print(f"Found similar formula in doc {other_block.source_obj.location[0]['input_path']}")
                                print(f"({formula}==={other_block.formulas[i]})\nSimilar keywords: {similar_words}")
                                print(f"({other_block.source_obj.location[0]['input_path']}) Similar keywords: {similar_words}")
                            found = True
                            break
                    if found:
                        break
                if found:
                        break
    return sim_doc_data_blocks, sim_doc_data_blocks_prob, sim_doc_data_blocks_test

similar_docs(test_doc_data, docs_data)

Processing block 1 of 6
Found similar formula in doc ../../data/do_not_upload/01-3-5-3.pdf
(F(x,u)=(b,u)+(c_{0}-A_{0}^{T}u,x)-x^{T}Q_{u}x.===(V_{2}f)(t)=(f(t),f(-t)),\quad t\in{\bf R}_{+}.)
Similar keywords: ['утверждение символах']
(../../data/do_not_upload/01-3-5-3.pdf) Similar keywords: ['утверждение символах']
Processing block 2 of 6
Processing block 3 of 6
Processing block 4 of 6
Processing block 5 of 6
Found similar formula in doc ../../data/do_not_upload/06-3-5-3.pdf
(P:\max\{f_{0}(x)\mid f_{j}(x)\leq0,j=1,\ldots,m\}===\mathrm{r}\equiv(b-a)^{\beta}:=\prod_{j=1}^{p}(b_{j}-a_{j})^{\beta_{j}};\quad|k|^{\beta}:=|k_{1}|^{\beta_{1}}\ldots|k_{p}|^{\beta_{p}};\quad(o)^{\beta_{j}}=1,\quad1\leqslant j\leqslant p.\quad\mathrm{Orc}\quad\mathrm{na})
Similar keywords: ['пересечение g_']
(../../data/do_not_upload/06-3-5-3.pdf) Similar keywords: ['пересечение g_']
Found similar formula in doc ../../data/do_not_upload/06-3-5-3.pdf
(P^{*}:\operatorname*{m i n}\{F(x,u)\mid\bigtriangledown_{x}F(x,u

([Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/01-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/01-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/04-12.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/01-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/07-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/03-3-5-3.pdf.,
  Block of ../../data/do_not_uploa

In [21]:
docs_data[1].location[0]['input_path']

'../../data/do_not_upload/07-3-5-3.pdf'

In [22]:
similar_docs(docs_data[1], docs_data)

Processing block 1 of 6
Processing block 2 of 6
Found similar formula in doc ../../data/do_not_upload/06-3-5-3.pdf
(\omega(\zeta)=\frac{h(\overline{{\zeta}})\wedge d\zeta}{g(\zeta,\overline{{\zeta}})},===(2\pi)^{|\beta|_{p}}|V_{k}|\leqslant\frac{(b-a)^{\beta}}{|k|^{\beta}}\sup\{|V_{1}^{(\beta)}(X)|:X\in T_{a,b}^{p}\})
Similar keywords: ['пересечение g_']
(../../data/do_not_upload/06-3-5-3.pdf) Similar keywords: ['пересечение g_']
Found similar formula in doc ../../data/do_not_upload/06-3-5-3.pdf
(f(z)=\frac{1}{C}\int_{\mu^{-1}(\rho)}f(\zeta)\omega(\zeta-z),===\lim_{k\to\infty}\left|f(X_{k,1})-f(X_{k,2})\right|=\exp\left[-\frac{1}{\delta^2-\rho^2(\gamma,\beta)}\right]>0.)
Similar keywords: ['пересечение g_']
(../../data/do_not_upload/06-3-5-3.pdf) Similar keywords: ['пересечение g_']
Processing block 3 of 6
Processing block 4 of 6
Found similar formula in doc ../../data/do_not_upload/06-3-5-3.pdf
(h(\zeta)=\sideset{}{'}\sum_{J}v_{J}\zeta[J]d\zeta_{J}===V(X)\sim\sum_{|k|_{p}=0}^{\infty}V

([Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/03-12.pdf.,
  Block of ../../data/do_not_upload/06-3-5-3.pdf.,
  Block of ../../da